# Comparação dos dois trabalhos sobre o mesmo dataset cervical

Duas análises independentes do **mesmo agregado de 10 BioProjects** de microbioma cervicovaginal:

1. **Tabela da coautora Paola Gyuliane Gonçalves** (`feature-table.txt`, 997 amostras) — a base usada no
   artigo, processada com DADA2/SILVA completo, classificação por espécie, sem filtro de profundidade.
2. **Pipeline próprio da autora** (889 amostras) — reconstrução independente desde os FASTQ brutos:
   deblur por região, classificador SILVA retreinado localmente, colapso por gênero, corte de 1.000 leituras.

A comparação abaixo testa se as **conclusões** do artigo se sustentam sob as duas tabelas, e documenta
onde elas divergem. Material completo em `analysis/` (scripts `07_compara_pipelines.py`, `01_auditoria.py`).

## 1. Valores publicados × pipeline próprio (Tabela 3)

| Métrica | Valor publicado (artigo) | Pipeline próprio |
|---|---|---|
| Amostras finais (de 1.042 catalogadas) | 997 | 889 (89,2%; teto de 937 sem o P21) |
| Variância por condição clínica (R² marginal) | 13,5% combinado (cond.+local.+proj.) | 0,59% só condição |
| Variância por projeto (R² marginal) | não isolado | 9,19% (≈15,5× a condição) |
| Localidade | no modelo combinado | colinear com projeto (0 g.l.) |
| GLLVM, variância por fator | não reportado | LV1/LV2 dominados por projeto (η² 56,7% / 58,2%) |
| Controle vs. câncer | 5,0% (F=38,3) | 6,52% (F=43,4; n=624) |
| HSIL vs. câncer | 3,6% | 5,75% (n=454) |
| LSIL vs. câncer | 2,2% | 4,44% (n=409) |
| HSIL vs. LSIL | n.s. (p=0,067) | n.s. (p=0,107; n=265) |
| RF macro-AUC, CV aleatória × LOPO | não reportado | 0,811 → 0,6 |
| Controle negativo (acurácia ao prever projeto) | não reportado | 82,79% |

As magnitudes absolutas diferem (o pipeline próprio parte de 889 das 997 amostras e reprocessa tudo do
zero), mas o **padrão qualitativo é o mesmo**, e os contrastes par a par reproduzem de perto o publicado,
inclusive acertando qual par não é significativo.

![Contrastes par a par: publicado × próprio](../analysis/figuras/f07_permanova.png)

## 2. Mesmas análises nas duas tabelas (Tabela 5)

Rodando exatamente o mesmo procedimento nas duas tabelas (pipeline próprio com o P22 reprocessado):

| Análise | Paola | Pipeline próprio |
|---|---|---|
| RF, CV aleatória (câncer / controle / HSIL / LSIL) | 0,97 / 0,85 / 0,76 / 0,74 | 0,96 / 0,82 / 0,74 / 0,71 |
| RF, só o projeto como preditor | 0,95 / 0,85 / 0,77 / 0,79 | 0,94 / 0,81 / 0,75 / 0,76 |
| RF, leave-one-project-out | 0,70 / 0,51 / 0,46 / 0,34 | 0,64 / 0,44 / 0,48 / 0,35 |
| Acurácia ao prever o projeto (acaso) | 88,1% (31,8%) | 86,3% (35,2%) |
| PERMANOVA: classe / projeto / classe além do projeto | 7,1% / 16,4% / 0,4% | 6,6% / 14,5% / 0,4% |
| Shannon câncer − controle: sem / com ajuste por projeto | +0,97 (p<10⁻²⁶) / +0,09 (p=0,43) | +1,00 / +0,14 (p=0,23) |
| 30 combinações classificador, câncer × controle (mediana) | CV 0,96; LOPO 0,70 | CV 0,95; LOPO 0,64 |

**Com o mesmo esquema de validação, as duas tabelas ficam a 0,01–0,03 de AUC uma da outra**, e a
correlação nos gêneros principais vai de 0,97 a 1,00. As diferenças de método mudam pouco; o que muda
as conclusões é **quais amostras e projetos entram**.

![Concordância entre os dois pipelines](../analysis/figuras/f11_concordancia_pipelines.png)

![Matrizes de confusão](../analysis/figuras/f04_matrizes_confusao.png)

## 3. O viés que as duas tabelas compartilham

A auditoria mostra que os principais biomarcadores do artigo vêm, cada um, de praticamente **um único
projeto**: *Burkholderia-Caballeronia-Paraburkholderia* (96% no P21, só controles), *Wolbachia* (100% no
P1), *Aeribacillus* e *Halomonas* (quase 100% no P9). O sinal clínico que se repete entre projetos é a
perda de *Lactobacillus* e o ganho de *Fusobacterium* e *Prevotella*; o único projeto onde o sinal de
câncer transfere para outros é o P19 (Nigéria, AUC ≈ 0,7–0,8).

O AUC ≈ 1 do artigo só aparece avaliando no próprio treino. Um modelo que recebe **apenas o código do
projeto** (sem nenhum dado de microbioma) empata com o microbioma inteiro — a assinatura de lote é o
que o classificador aprende.

![Origem dos biomarcadores por projeto](../analysis/figuras/f15_origem_biomarcadores.png)

![Meta-análise câncer](../analysis/figuras/f17_meta_cancer.png)

## 4. Incongruências entre as duas tabelas (Tabela 6)

| Ponto | Paola (artigo) | Pipeline próprio | A corrigir |
|---|---|---|---|
| P22 | classificado corretamente | processado como V4; 79% sem gênero (reprocessado: Bray-Curtis 0,02) | reprocessado, falta SILVA V3-V4 |
| P21 (Espanha) | incluído: 103 controles + 2 HSIL | excluído (primer não publicado) | decisão conjunta |
| Amostras < 1.000 leituras | 26 mantidas | filtradas | Paola |
| P16 | 29 amostras; ~48% Unassigned+Eukaryota | 47 amostras; metadados 55, artigo 26 | Paola |
| P14, 21 runs não-16S (DNA humano) | 5 com 2–4 leituras, 16 ausentes | descartadas pelo cutadapt | registrar |
| P14 (Brasil) | 13 HSIL / 14 controles | 11 / 11 (artigo: 22 / 21) | as duas |
| Total | 997 na tabela, 1.008 no artigo | 889 | Paola |
| Método | DADA2 (texto) / Deblur (pasta), espécie, sem filtro | Deblur por região, gênero, filtro 1.000 | escolha de método |

**O que muda as conclusões não é o método, e sim quais amostras e projetos entram** (P21, P16 e as
amostras de baixa profundidade).

![Contabilidade de amostras](../analysis/figuras/f12_contabilidade_amostras.png)

## Conclusão da comparação

As duas análises convergem no essencial: sobre o mesmo dataset agregado, **o efeito de projeto de origem
domina a variação composicional e o desempenho de classificação colapsa sob validação
leave-one-project-out**. As diferenças numéricas entre os dois trabalhos vêm quase inteiramente das
decisões de inclusão de amostras/projetos, não do pipeline bioinformático em si. Essa robustez reforça a
tese central do artigo e orienta a dissertação (soja): o que precisa ser controlado é o confundimento com
o projeto de origem, não a escolha de denoiser ou classificador.